# R19-H209 - The shared-feature stratum: judging attribution scoping on a fair bench

**Round** R19 | **Hypothesis** H209 | **Graph** neo4j2 (READ-ONLY) | **Compute** CPU-only, deterministic, no LLM

H205's refutation branch fired: its residual errors are dominated by genuinely-shared features whose
"absent" labels on foreign products are contestable. Before the scorer can be judged, the bench needs an
explicit shared-feature stratum.

- **(a)** re-adjudicate the H205 attribution bench with an explicit shared-feature stratum (BLIND: each
  feature gold labelled exclusive-to-product or legitimately-shared from document evidence, frozen before
  scoring; shared features score present for ALL legitimate owners). The foreign-device-exclusion scoping
  variant must reach **>= 0.90 on the exclusive stratum** and **>= 0.97 combined**
- **(b)** raise the per-product render relation cap 15 -> 40; does STRICT scoping recover to within 3 pts
  of the exclusion variant?

**Refuted if** exclusive-feature errors persist after re-adjudication (scoping is then insufficient and
attribution needs graph-side provenance, not render-side scoping).

Harness is the pinned H207 canonical spec (render_fingerprint `96ab16d299fbbc71`, asserted below).
Note: H211's inert render-cap raise was on the seed-neighbourhood surface; H209's cap is the per-product
render section of the attribution bench - a different surface (distinction cited in the report).

## Setup - CPU-only, neo4j2 pinned (explicit driver, NOT the .env default) [DEF-5]

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
import re, json, pickle, hashlib, unicodedata, datetime
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
from neo4j import GraphDatabase
from dotenv import dotenv_values
from rich import print as rprint

ROOT = Path("..")
NEO4J2 = "bolt://user-konrad.jelen-kgf-neo4j2:7687"          # 172.19.0.9, READ-ONLY reference
_env = dotenv_values(ROOT / ".env")
PW = os.environ.get("NEO4J_PASSWORD") or _env.get("NEO4J_PASSWORD", "kgfoundry")
driver2 = GraphDatabase.driver(NEO4J2, auth=("neo4j", PW))
VEC = None  # set after settings load
rprint("[cyan]neo4j2 pinned, READ-ONLY, CPU-only[/cyan]")

neo4j2 pinned, READ-ONLY, CPU-only

## Graph pull + production-faithful render primitives (H207 harness; relation cap parametrised for clause b)

In [2]:
with driver2.session() as s:
    ents = s.run("MATCH (e:Entity) RETURN e.id AS id, e.name AS name, e.description AS description, "
                 "properties(e) AS props, labels(e) AS types").data()
    edges = s.run("MATCH (a:Entity)-[r]-(b:Entity) WHERE type(r)<>'SIMILAR_TO' AND a.id<b.id "
                  "RETURN DISTINCT a.id AS a, b.id AS b, type(r) AS rel").data()
    prop_rows = s.run("MATCH (p:Proposition)-[:ABOUT]->(e:Entity) RETURN e.id AS eid, p.text AS text").data()
    alias_rows = s.run("MATCH (e:Entity)-[:SAME_AS*1..2]-(a:Entity) WHERE e.id<>a.id "
                       "RETURN e.id AS eid, collect(DISTINCT a.id)[..5] AS aliases").data()
    emb_head = {r["id"]: r["head"] for r in s.run(
        "MATCH (e:Entity) WHERE e.embedding IS NOT NULL RETURN e.id AS id, e.embedding[0..8] AS head").data()}
    graph_docs = set(r["nm"] for r in s.run("MATCH (dd:KGFDocument) RETURN dd.name AS nm").data())
node = {r["id"]: r for r in ents}; names = {r["id"]: r["name"] for r in ents}
props_by = defaultdict(list); [props_by[r["eid"]].append(r["text"]) for r in prop_rows]
alias_by = {r["eid"]: r["aliases"] for r in alias_rows}
rels_by = defaultdict(list)
for e in edges:
    rels_by[e["a"]].append((e["rel"], e["b"])); rels_by[e["b"]].append((e["rel"], e["a"]))
def spec_of(r): return {k.removeprefix("prop_"): v for k, v in r["props"].items() if k.startswith("prop_")}
def merged_spec(nid):
    r = node[nid]; spec = dict(spec_of(r))
    for a in [a for a in alias_by.get(nid, []) if a in node]:
        for k, v in spec_of(node[a]).items(): spec.setdefault(k, v)
    return spec
def base_render(nid):
    r = node[nid]; spec = merged_spec(nid); al = [a for a in alias_by.get(nid, []) if a in node]
    aka = (f"Also known as: {', '.join(names.get(a, '') for a in al)}\n" if al else "")
    return f"## {r['name']} ({', '.join(r['types'])})\n{aka}{r['description'] or ''}\nProperties: {json.dumps(spec, default=str)}"
def seed_render(nid, cap=15):
    rels = "; ".join(f"{t} -> {names.get(b, '')}" for t, b in rels_by.get(nid, [])[:cap])
    return base_render(nid) + "\nRelations: " + rels
def units1(nid, cap=15): return [seed_render(nid, cap)] + props_by.get(nid, [])
_TM = dict.fromkeys(map(ord, "®™©"), None)
def gnorm(s):
    s = (s or "").translate(_TM); s = unicodedata.normalize("NFKC", s)
    s = s.replace(" ", " ").replace("×", "x").replace("*", "x").replace("·", "x")
    s = re.sub(r"(?<=\d),(?=\d)", "", s)
    return re.sub(r"\s+", " ", s.casefold()).strip()
rprint(f"[green]pulled neo4j2[/green] entities {len(node)} edges {len(edges)} props {len(prop_rows)} docs {len(graph_docs)}")

pulled neo4j2 entities 2798 edges 3905 props 19654 docs 27

## Fingerprint assertion - render_fingerprint MUST match the pinned H207 census before measuring

In [3]:
from knowledge_graph_foundry import load_settings
settings = load_settings(ROOT / "config.yml"); VEC = settings.graphrag.vector_index_name
CANON_SPEC = dict(
    source="pipeline._retrieve_local entity_blocks (production query path)",
    per_seed=["## name (types)", "Also known as (SAME_AS*1..2, <=5)", "description",
              "Properties: json(prop_* keys, alias-merged)", "Relations: type -> name (<=15, non-SIMILAR_TO)"],
    proposition_channel="per-seed attached propositions (Proposition-[:ABOUT]->seed)",
    eval_k=64, retrieve_top_k=128, rel_limit=15,
    seed_order="score desc, id asc (deterministic tie-break)",
    scorer="deterministic: exact_present(numeric/code) OR word_overlap>=0.6(prose); CPU-only, no NLI")
def render_fingerprint(spec, ids):
    blob = json.dumps({k: spec[k] for k in sorted(spec)}, default=str) + "\x1e" + \
           "\x1e".join(seed_render(c, 15) for c in sorted(ids))
    return hashlib.sha256(blob.encode()).hexdigest()[:16]
ALL_IDS = sorted(node)
RENDER_FP = render_fingerprint(CANON_SPEC, ALL_IDS)
CONTENT_HASH = hashlib.sha256(("\x1e".join(seed_render(c, 15) for c in ALL_IDS)).encode()).hexdigest()[:16]
PINNED = dict(render_fp="96ab16d299fbbc71", content_hash="6fdc41bde495d1a3")
assert RENDER_FP == PINNED["render_fp"], f"render fingerprint drift: {RENDER_FP}"
assert CONTENT_HASH == PINNED["content_hash"], f"graph content drift: {CONTENT_HASH}"
rprint(f"[magenta]render_fingerprint[/magenta] {RENDER_FP}  [magenta]content_hash[/magenta] {CONTENT_HASH}  -> [green]MATCH pinned H207[/green]")

2026-07-07 21:38:23.186 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


render_fingerprint 96ab16d299fbbc71  content_hash 6fdc41bde495d1a3  -> MATCH pinned H207

## Attribution bench - blocks, queried-product recovery, document ownership evidence

The bench is 24 golds x {pos_own, neg_foreign} = 48 pairs. Each block's queried product is recovered
from the H188 retrieval cache. Document evidence = whether the queried product's OWN home documentation
(from the parser-round text cache, 27-doc corpus) carries the feature.

In [4]:
# ---- attribution bench: blocks, queried-product recovery, home-doc ownership evidence ----
from knowledge_graph_foundry import load_settings
from knowledge_graph_foundry.graph.graphrag import vector_query
bench = json.load(open(ROOT / "data/processed/instrument-prose-bench-h196.json"))["pairs"]
for p in bench: p["label"] = int(p["label"])
blocks = defaultdict(list)
for p in bench: blocks[tuple(p["ctx_ids"])].append(p)

wqc = pickle.load(open(ROOT / "notebooks/.wide_probes_h188_qcache.pkl", "rb"))
def wqk(q): return hashlib.md5(q.encode()).hexdigest()
allp = (json.load(open(ROOT / "data/processed/probes-wide-h188b.json"))["probes"]
        + json.load(open(ROOT / "data/processed/probes-wide-h188.json"))["probes"])
blockprod = {}
for p in allp:
    if wqk(p["question"]) not in wqc: continue
    ids = tuple(x["id"] for x in vector_query(driver2, wqc[wqk(p["question"])], VEC, top_k=64) if x["id"] in node)[:8]
    if ids in blocks and ids not in blockprod: blockprod[ids] = p.get("product")
prose_prod = {"constant lower pressure": "DreamStation", "AutoSet for Her": "ResMed AirSense 10 AutoSet for Her",
 "gradually acclimate": "DreamStation CPAP Pro", "amplitude of oscillations": "Seattle-PAP",
 "sleep onset detection": "AutoRamp", "reduces the pressure during expiration": "ResMed AirSense 11"}
for k, ps in blocks.items():
    if k not in blockprod:
        po = [p for p in ps if p["kind"] == "pos_own"][0]; blockprod[k] = prose_prod.get(po["gold"])
rprint(f"blocks {len(blocks)}  queried-product recovered {sum(1 for k in blocks if blockprod.get(k))}/{len(blocks)}")

# document evidence: does a queried product's OWN home documentation carry the feature?
PC = ROOT / "reports/parser-round-cache"
texts = {pr: json.loads((PC / f"text_{pr}.json").read_text()) for pr in ["docling", "pymupdf4llm", "pdfplumber"]}
DOCS = [dn for dn in texts["docling"].keys() if dn in graph_docs]
rawd = {dn: gnorm("\n".join(texts[pr].get(dn, "") for pr in ["pdfplumber", "pymupdf4llm", "docling"])) for dn in DOCS}
def _n(s): return re.sub(r"\s+", " ", (s or "").casefold())
GEN = set("cpap auto pro plus device machine system for her the and with card sd oxygen concentrator portable stationary medical drive heavy duty precision easy superstar respiratory sleep standard series brochure user manual datasheet guide care solutions catalog products product bipap apap adjustable heated humidifier tube nasal mask pillows full face fit pack elite lite".split())
def toks(s): return set(re.findall(r"[a-z0-9]+", s.lower()))
title = {dn: _n(texts["docling"].get(dn, "")[:300]) for dn in DOCS}
def keyset(nm): return {k for k in (toks(nm) - GEN) if len(k) >= 4}
def home_docs(nm):
    ks = keyset(nm)
    return sorted({dn for dn in DOCS if (ks & toks(dn)) or (_n(nm) in title[dn])}) if ks else []
def owns_in_homedoc(prod, gold): return any(gnorm(gold) in rawd[dn] for dn in home_docs(prod or ""))
rprint(f"corpus docs with text: {len(DOCS)}/27")


blocks 24  queried-product recovered 24/24

corpus docs with text: 27/27

## Clause (a) - frozen blind taxonomy + re-adjudicated labels

Each of the 24 feature/prose golds is committed as exclusive-to-brand (proprietary/trademarked term or
single-product prose) or legitimately-shared (generic capability), from document + domain evidence, BEFORE
any scorer runs. neg_foreign labels flip 0 -> 1 only where document evidence shows the queried product
legitimately owns the feature (present in its home docs); no 1 -> 0 flips.

In [5]:
# ---- FROZEN BLIND TAXONOMY (committed from document + domain evidence BEFORE scoring) ----
# exclusive = brand-proprietary/trademarked or single-product-specific prose;
# shared     = generic capability multiple brands legitimately provide (score present for ALL legitimate owners).
EXCLUSIVE = {"AutoRamp", "AutoSet for Her", "SmartStart", "EPR", "Climate Control", "C-Flex",
             "Opti-Start", "EZ-Start", "CPAP-Check", "System One resistance control",
             "amplitude of oscillations", "constant lower pressure"}
SHARED = {"heated humidifier", "adjustable humidification", "adjustable ramp time", "wireless data transfer",
          "integrated cellular modem", "Wi-Fi connectivity", "Bluetooth connectivity", "mask fit check",
          "Auto-CPAP mode", "reduces the pressure during expiration", "gradually acclimate", "sleep onset detection"}
def stratum_of(g): return "exclusive" if g in EXCLUSIVE else ("shared" if g in SHARED else "?")

# re-adjudicated labels: keep pos_own (H196); flip neg_foreign 0->1 ONLY where document evidence shows
# the queried product legitimately owns the feature (owns_in_homedoc). No 1->0 flips.
readj = []
for k, ps in blocks.items():
    prod = blockprod.get(k)
    for p in ps:
        lab = p["label"]
        if p["kind"] == "neg_foreign" and lab == 0 and owns_in_homedoc(prod, p["gold"]):
            lab = 1
        readj.append(dict(ctx=k, prod=prod, gold=p["gold"], kind=p["kind"], stratum=stratum_of(p["gold"]),
                          orig_label=p["label"], label=lab))
label_flips = [r for r in readj if r["orig_label"] != r["label"]]
lab_by = {(r["ctx"], r["gold"], r["kind"]): r["label"] for r in readj}
strat_by = {(r["ctx"], r["gold"], r["kind"]): r["stratum"] for r in readj}
rprint(f"[bold]frozen taxonomy[/bold] exclusive golds={len(EXCLUSIVE)} shared golds={len(SHARED)}  "
       f"| document-evidence label flips (0->1): {len(label_flips)}")
for r in label_flips:
    rprint(f"   [{r['stratum']}] {r['gold']!r} foreignQ={str(r['prod'])[:26]} 0->1 (owns in home doc)")

# ---- scorer + scope variants (H205 verbatim logic) ----
STOPW = {"the","a","an","of","for","and","or","to","in","on","with","is","are","be","that","this","it","its",
 "does","do","offer","offers","what","which","support","supported","supports","feature","features",
 "have","has","your","you","during","from","by","as","at","per","up","down","her","his","their","not"}
def _wset(t): return set(re.findall(r"[a-z][a-z0-9\-]{2,}", gnorm(t)))
def wov(gold, ids, cap):
    w = _wset(gold) - STOPW
    cw = _wset(" ".join(t for n in ids for t in units1(n, cap)))
    return bool(w) and len(w & cw) / len(w) >= 0.6
nm2id = defaultdict(list)
for nid, nm in names.items(): nm2id[gnorm(nm)].append(nid)
DEVT = {"CPAPDevice", "ProductModel", "Device", "Product"}
def resolveP(pn):
    if not pn: return None
    g = gnorm(pn)
    if g in nm2id: return nm2id[g][0]
    c = [nid for nid, nm in names.items() if g in gnorm(nm) or gnorm(nm) in g]
    c.sort(key=lambda i: (-(len(DEVT & set(node[i]["types"]))), abs(len(names[i]) - len(pn))))
    return c[0] if c else None
def qP(k): return resolveP(blockprod.get(k))
def is_dev(e): return bool(DEVT & set(node[e]["types"]))
def scope_full(k, P): return list(k)
def scope_strict(k, P): return ([P] + [a for a in alias_by.get(P, []) if a in node]) if P else list(k)
def scope_nofdev(k, P):
    keep = ({P} | set(a for a in alias_by.get(P, []) if a in node)) if P else set()
    return [e for e in k if (not is_dev(e)) or (e in keep)]

def run(scope, cap, labmap=None):
    labmap = labmap or lab_by
    per = defaultdict(list); errs = []
    for k, ps in blocks.items():
        P = qP(k); sc = scope(k, P)
        for p in ps:
            key = (k, p["gold"], p["kind"]); lab = labmap[key]; strat = strat_by[key]
            pred = int(wov(p["gold"], sc, cap)) if sc else 0
            ok = (pred == lab)
            per[strat].append(ok); per["ALL"].append(ok)
            if not ok:
                errs.append(dict(stratum=strat, kind=p["kind"], gold=p["gold"],
                                 prod=names.get(P, "?"), label=lab, pred=pred, ctx=k, P=P))
    res = {kk: round(float(np.mean(vv)), 3) for kk, vv in per.items() if kk in ("exclusive", "shared", "ALL")}
    return res, errs
def combined(all_acc): return round((49 * 1.0 + len(bench) * all_acc) / (49 + len(bench)), 3)


frozen taxonomy exclusive golds=12 shared golds=12  | document-evidence label flips (0->1): 6

'EZ-Start' foreignQ=DreamStation 0->1 (owns in home doc)

'EPR' foreignQ=ResMed AirSense 10 AutoSet 0->1 (owns in home doc)

'gradually acclimate' foreignQ=DreamStation Auto CPAP wit 0->1 (owns in home doc)

'heated humidifier' foreignQ=AirSense 10 Elite 0->1 (owns in home doc)

'C-Flex' foreignQ=DreamStation Auto CPAP wit 0->1 (owns in home doc)

'CPAP-Check' foreignQ=DreamStation 0->1 (owns in home doc)

### Harness fidelity - reproduce H205's numbers on the ORIGINAL labels (prose/feature strata)

In [6]:
def run_orig(scope, cap):
    per = defaultdict(list)
    for k, ps in blocks.items():
        P = qP(k); sc = scope(k, P)
        for p in ps:
            pred = int(wov(p["gold"], sc, cap)) if sc else 0
            per[p["stratum"]].append(pred == p["label"]); per["ALL"].append(pred == p["label"])
    return {kk: round(float(np.mean(vv)), 3) for kk, vv in per.items()}
_r = run_orig(scope_nofdev, 15)
rprint(f"no_foreign_dev on ORIGINAL labels: prose={_r.get('prose')} feature={_r.get('feature')} "
       f"combined={combined(_r['ALL'])}  [dim](H205 ref: 0.917 / 0.833 / 0.928)[/dim]")
assert _r.get('prose') == 0.917 and _r.get('feature') == 0.833, "harness fidelity drift vs H205"
rprint("[green]harness reproduces H205 exactly[/green]")

no_foreign_dev on ORIGINAL labels: prose=0.917 feature=0.833 combined=0.928  (H205 ref: 0.917 / 0.833 / 0.928)

harness reproduces H205 exactly

### Clause (a) scoring - re-adjudicated labels, exclusive stratum + combined bars

In [7]:
results = {}
for cap in (15, 40):
    for nmv, sc in [("baseline_full", scope_full), ("scoped_strict", scope_strict), ("scoped_no_foreign_dev", scope_nofdev)]:
        res, errs = run(sc, cap)
        results[(nmv, cap)] = dict(res=res, combined=combined(res["ALL"]), errs=errs)
        rprint(f"cap{cap:2d} {nmv:22s} exclusive={res.get('exclusive')} shared={res.get('shared')} "
               f"ALL={res['ALL']} combined={combined(res['ALL'])} errs={len(errs)}")
PRIMARY = results[("scoped_no_foreign_dev", 15)]
excl = PRIMARY["res"]["exclusive"]; comb = PRIMARY["combined"]
bar_excl = excl >= 0.90; bar_comb = comb >= 0.97
rprint(f"\n[bold]clause (a) bars[/bold]: exclusive>=0.90 = [{'green' if bar_excl else 'red'}]{bar_excl}[/] ({excl})  "
       f"| combined>=0.97 = [{'green' if bar_comb else 'red'}]{bar_comb}[/] ({comb})")
rprint("[bold]persisting exclusive-stratum errors (no_foreign_dev, cap15):[/bold]")
for e in [e for e in PRIMARY["errs"] if e["stratum"] == "exclusive"]:
    rprint(f"   {e['gold']!r} prod={e['prod'][:22]} kind={e['kind']} L={e['label']} P={e['pred']}")

cap15 baseline_full          exclusive=0.875 shared=0.75 ALL=0.812 combined=0.907 errs=9

cap15 scoped_strict          exclusive=0.667 shared=0.542 ALL=0.604 combined=0.804 errs=19

cap15 scoped_no_foreign_dev  exclusive=0.833 shared=0.792 ALL=0.812 combined=0.907 errs=9

cap40 baseline_full          exclusive=0.875 shared=0.75 ALL=0.812 combined=0.907 errs=9

cap40 scoped_strict          exclusive=0.667 shared=0.542 ALL=0.604 combined=0.804 errs=19

...

persisting exclusive-stratum errors (no_foreign_dev, cap15):

'constant lower pressure' prod=DreamStation kind=pos_own L=0 P=1

'EZ-Start' prod=DreamStation kind=neg_foreign L=1 P=0

'C-Flex' prod=DreamStation Auto CPAP kind=neg_foreign L=1 P=0

'CPAP-Check' prod=DreamStation kind=neg_foreign L=1 P=0

### Provenance diagnostic - are the persisting exclusive errors graph-side?

For each persisting exclusive error, check whether the gold token appears ANYWHERE in the full retrieved
block vs only in the scoped renders vs nowhere. "nowhere / on a dropped foreign device" = the feature is
legitimately owned (per docs) but the graph never surfaces it on the product - a graph-side provenance gap
that render-side scoping cannot fix.

In [8]:
prov = []
for e in [e for e in PRIMARY["errs"] if e["stratum"] == "exclusive" and e["label"] == 1 and e["pred"] == 0]:
    k = e["ctx"]; P = e["P"]; g = gnorm(e["gold"])
    in_full = any(g in gnorm(" ".join(units1(n, 15))) for n in k)
    in_scoped = any(g in gnorm(" ".join(units1(n, 15))) for n in scope_nofdev(k, P))
    in_prod = P is not None and g in gnorm(" ".join(units1(P, 40)))
    cause = ("in-block-but-on-dropped-device" if in_full and not in_scoped
             else ("absent-from-block (retrieval/extraction gap)" if not in_full
                   else "in-scope-but-below-overlap-thr"))
    prov.append(dict(gold=e["gold"], prod=e["prod"], in_full_block=in_full, in_scoped=in_scoped,
                     in_product_render_cap40=in_prod, cause=cause))
    rprint(f"   {e['gold']!r} prod={e['prod'][:22]}: {cause}  (in_full={in_full} in_scoped={in_scoped} in_prod_cap40={in_prod})")
rprint(f"\ngraph-side provenance gaps among persisting exclusive errors: {len(prov)}")

'EZ-Start' prod=DreamStation: absent-from-block (retrieval/extraction gap)  (in_full=False in_scoped=False 
in_prod_cap40=False)

'C-Flex' prod=DreamStation Auto CPAP: absent-from-block (retrieval/extraction gap)  (in_full=False 
in_scoped=False in_prod_cap40=False)

'CPAP-Check' prod=DreamStation: in-block-but-on-dropped-device  (in_full=True in_scoped=False 
in_prod_cap40=False)

graph-side provenance gaps among persisting exclusive errors: 3

## Clause (b) - render cap 15 -> 40: does STRICT scoping recover to within 3 pts of exclusion?

In [9]:
excl15 = results[("scoped_no_foreign_dev", 15)]["res"]["ALL"]
for cap in (15, 40):
    strict = results[("scoped_strict", cap)]["res"]["ALL"]
    nofdev = results[("scoped_no_foreign_dev", cap)]["res"]["ALL"]
    rprint(f"cap{cap}: strict ALL={strict}  no_foreign_dev ALL={nofdev}  gap={round((nofdev-strict)*100,1)} pts")
strict40 = results[("scoped_strict", 40)]["res"]["ALL"]
nofdev40 = results[("scoped_no_foreign_dev", 40)]["res"]["ALL"]
gap40 = round((nofdev40 - strict40) * 100, 1)
cap_inert = (results[("scoped_no_foreign_dev", 40)]["res"] == results[("scoped_no_foreign_dev", 15)]["res"]
             and results[("scoped_strict", 40)]["res"] == results[("scoped_strict", 15)]["res"])
bar_b = gap40 <= 3.0
rprint(f"\n[bold]clause (b)[/bold]: strict within 3 pts of exclusion at cap40 = [{'green' if bar_b else 'red'}]{bar_b}[/] "
       f"(gap {gap40} pts)  | render-cap raise inert = {cap_inert}")
rprint("[dim]distinction: H211's inert cap was the seed-neighbourhood render; here the cap is the per-product "
       "render section - a different surface. Both inert, but here because the legitimately-owned features are "
       "not in the product's relation tails at any cap (extraction never attached them).[/dim]")

cap15: strict ALL=0.604  no_foreign_dev ALL=0.812  gap=20.8 pts

cap40: strict ALL=0.604  no_foreign_dev ALL=0.812  gap=20.8 pts

clause (b): strict within 3 pts of exclusion at cap40 = False (gap 20.8 pts)  | render-cap raise inert = True

distinction: H211's inert cap was the seed-neighbourhood render; here the cap is the per-product render section - a
different surface. Both inert, but here because the legitimately-owned features are not in the product's relation 
tails at any cap (extraction never attached them).

## Machine-readable report + verdict

In [10]:
ts = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
# flip-sensitivity: exclusive/combined if the 3 intra-brand exclusive flips are NOT applied
INTRA = {"EZ-Start", "C-Flex", "CPAP-Check"}
lab_noflip = {}
for r in readj:
    lab = r["label"]
    if r["kind"] == "neg_foreign" and r["gold"] in INTRA and r["orig_label"] == 0: lab = 0
    lab_noflip[(r["ctx"], r["gold"], r["kind"])] = lab
res_nf, _ = run(scope_nofdev, 15, lab_noflip)
report = dict(
    round="R19", hypothesis="H209", utc=ts, graph="neo4j2",
    render_fingerprint=RENDER_FP, content_hash=CONTENT_HASH, compute="CPU-only, deterministic, no LLM",
    harness_fidelity="no_foreign_dev on original labels reproduces H205 (prose 0.917 / feature 0.833 / combined 0.928)",
    taxonomy=dict(exclusive=sorted(EXCLUSIVE), shared=sorted(SHARED),
                  label_flips=[dict(gold=r["gold"], stratum=r["stratum"], foreign_product=r["prod"]) for r in label_flips]),
    clause_a=dict(
        variants={f"{n}_cap{c}": dict(exclusive=v["res"].get("exclusive"), shared=v["res"].get("shared"),
                                      all=v["res"]["ALL"], combined=v["combined"])
                  for (n, c), v in results.items()},
        primary="scoped_no_foreign_dev_cap15",
        exclusive_stratum=excl, combined=comb,
        bar_exclusive_ge_090=bool(bar_excl), bar_combined_ge_097=bool(bar_comb),
        persisting_exclusive_errors=[dict(gold=e["gold"], prod=e["prod"], kind=e["kind"], label=e["label"], pred=e["pred"])
                                     for e in PRIMARY["errs"] if e["stratum"] == "exclusive"],
        provenance_gaps=prov,
        flip_sensitivity=dict(no_intrabrand_flip_exclusive=res_nf["exclusive"], no_intrabrand_flip_combined=combined(res_nf["ALL"]),
                              note="combined>=0.97 needs bench ALL>=0.939; best observed ALL<=0.875 -> combined bar unmet under every adjudication")),
    clause_b=dict(cap_raise="15->40", render_cap_inert=bool(cap_inert),
                  strict_all_cap40=strict40, exclusion_all_cap40=nofdev40, gap_pts=gap40, within_3pts=bool(bar_b),
                  h211_distinction="H211 inert cap = seed-neighbourhood render; H209 cap = per-product render section (different surface); both inert, here because legitimately-owned features are absent from the product's relation tails at any cap"),
    refuted=True,
    verdict_recommendation="REFUTED",
    verdict_basis=("clause (a) not met under any adjudication: combined>=0.97 robustly unmet (0.907-0.938; needs bench "
                   "ALL>=0.939 vs observed 0.812-0.875), and under the committed document-evidence adjudication the "
                   "exclusive stratum is 0.833<0.90 with persisting errors dominated by graph-side provenance gaps "
                   "(EZ-Start/C-Flex/CPAP-Check legitimately owned by DreamStation per its own docs but never surfaced "
                   "on the product in the graph). clause (b): render-cap raise inert, strict scoping stays ~21 pts below "
                   "exclusion. Attribution needs graph-side provenance, not render-side scoping - exactly the registered refuter."))
out = ROOT / f"reports/shared-feature-h209-{ts}.json"
out.write_text(json.dumps(report, indent=2, default=str))
rprint(f"[green]wrote[/green] {out}")
rprint(f"[bold]VERDICT[/bold] {report['verdict_recommendation']}  "
       f"(exclusive={excl} combined={comb} clause_b_gap={gap40}pts cap_inert={cap_inert})")
driver2.close()

/tmp/ipykernel_2770284/205231085.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


wrote ../reports/shared-feature-h209-20260707T193825Z.json

VERDICT REFUTED  (exclusive=0.833 combined=0.907 clause_b_gap=20.8pts cap_inert=True)